# Chapter 15 &mdash; The Post Correspondence Problem

**Concept 1 of the Chapter 15 decomposition:** *The Post Correspondence Problem: the Drosophila of Computability*

Match dominoes so the top string equals the bottom string &mdash; the Drosophila of computability.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-The-PCP/Concept-The-PCP.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A **PCP instance** is a finite list of **dominoes**, each with a **top** and a
**bottom** string. A **solution** is a non-empty sequence of dominoes (repeats
allowed) whose concatenated tops equal its concatenated bottoms.

$$\left[\frac{b}{ca}\right]\ \left[\frac{a}{ab}\right]\ \left[\frac{ca}{a}\right]\ \left[\frac{abc}{c}\right]$$

It looks like a puzzle, and it is &mdash; but it is **undecidable**, and it reduces to a
remarkable number of other problems. That combination of simplicity and reach is why
Post's problem is the **Drosophila** of computability: small enough to experiment on,
rich enough to prove things with.

Two things make search hard: solutions can be **very long**, and there is **no bound**
to search up to.

## 2. Definitions

### The solver

In [ ]:
# --- a Post Correspondence solver, bounded by tile count ----------------
# An instance is a list of (top, bottom) dominoes.  A solution is a
# non-empty sequence of indices whose concatenated tops equal its bottoms.
def pcp_search(tiles, maxlen=8):
    from collections import deque
    # a partial solution is (indices, top, bottom); one side is a prefix
    # of the other, or the partial is dead
    dq = deque([((i,), t, b) for i, (t, b) in enumerate(tiles)])
    while dq:
        idx, top, bot = dq.popleft()
        if top == bot:
            return list(idx)
        if len(idx) >= maxlen:
            continue
        if not (top.startswith(bot) or bot.startswith(top)):
            continue                                  # dead: they diverge
        for j, (t, b) in enumerate(tiles):
            dq.append((idx + (j,), top + t, bot + b))
    return None

def pcp_check(tiles, sol):
    top = ''.join(tiles[i][0] for i in sol)
    bot = ''.join(tiles[i][1] for i in sol)
    return top == bot, top, bot

def show_tiles(tiles):
    print("   " + "  ".join("[%s/%s]" % t for t in tiles))

### Two classic instances

In [ ]:
CLASSIC = [('b', 'ca'), ('a', 'ab'), ('ca', 'a'), ('abc', 'c')]
NOSOL   = [('ab', 'aba'), ('bb', 'aa'), ('aba', 'bb')]

## 3. Tests

A solvable instance, and its solution.

In [ ]:
show_tiles(CLASSIC)
sol = pcp_search(CLASSIC, maxlen=6)
print("solution indices :", sol)
ok, top, bot = pcp_check(CLASSIC, sol)
print("top    :", top)
print("bottom :", bot)
assert ok

Reading the solution as a row of dominoes.

In [ ]:
print("  tops   : " + " | ".join(CLASSIC[i][0] for i in sol))
print("  bottoms: " + " | ".join(CLASSIC[i][1] for i in sol))
print("  both concatenate to %r" % top)

A partial match must keep one side a **prefix** of the other.

In [ ]:
partials = [((0,), 'b', 'ca'), ((0, 1), 'ba', 'caab'), ((1,), 'a', 'ab')]
for idx, t, b in partials:
    alive = t.startswith(b) or b.startswith(t)
    print("  %-10s top %-6r bot %-6r  still alive? %s" % (str(idx), t, b, alive))
print("\nOnce neither is a prefix of the other, no extension can rescue it.")

An instance with no short solution &mdash; and you cannot tell if it has a long one.

In [ ]:
for m in [4, 6, 8]:
    print("  searched to length %d : %s" % (m, pcp_search(NOSOL, maxlen=m)))
assert pcp_search(NOSOL, maxlen=8) is None
print("\nNo solution found.  Is there none, or is it longer than 8?")

Solutions can be **surprisingly long**, which is why 'search a bit more' fails.

In [ ]:
HARD = [('001', '0'), ('01', '011'), ('01', '101'), ('10', '001')]
show_tiles(HARD)
for m in [4, 6, 8]:
    s = pcp_search(HARD, maxlen=m)
    print("  to length %d : %s" % (m, s))
print("\nKnown PCP instances with four dominoes need solutions of length > 200.")

## 4. Exercises


1. Find a solution to `[b/ca][a/ab][ca/a][abc/c]` by hand before looking.
2. Is PCP with **one** domino decidable? With dominoes over a **one-letter** alphabet?
3. Why does the prefix test prune so effectively, and why is it not enough?

In [ ]:
# Your work for the exercises above.